# Week 2 Day 2 — Save / Load Checkpoints
**Jul 9, 2026**

Every notebook so far has trained a model and then let it evaporate when the kernel restarts. Today: save training progress to disk and resume from it — including the pieces that are easy to forget (optimizer state, epoch number, best score so far), not just the model's weights.

Scaffold as usual: TODO stubs, hints not formulas, self-check cells.

## Part 1: Data and model

Given. Same style of synthetic binary classification data as recent days, plus a small MLP — the point today is the save/load mechanics, not the model.

In [1]:
import torch
import torch.nn as nn
from pathlib import Path

torch.manual_seed(0)
n_samples, n_features = 400, 4
X = torch.randn(n_samples, n_features)
true_w = torch.tensor([0.8, -1.2, 0.5, 1.0])
y = (X @ true_w > 0).float()

model = nn.Sequential(nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.BCEWithLogitsLoss()

ckpt_dir = Path("checkpoints")
ckpt_dir.mkdir(exist_ok=True)

## Part 2: What goes in a checkpoint

TODO: write `save_checkpoint(path, model, optimizer, epoch, best_val_loss)`.

A checkpoint should be a single dict with (at least) these four keys:

- `model_state_dict` — not the model object itself. `nn.Module` has a method that returns just its learnable parameters as an ordered dict of tensors, decoupled from the class definition. Saving the object directly would pickle your `MLPClassifier`/`nn.Sequential` class code along with the weights, which breaks if that class definition ever changes later — the state dict alone is portable.
- `optimizer_state_dict` — same idea, but for the optimizer. Why does this matter for `Adam` specifically, more than it would for plain `SGD`? (Think about what extra values Adam tracks per-parameter beyond the gradient itself, and what "resuming" training would mean if those were reset to zero.)
- `epoch` — a plain int, so a resumed run knows where to continue the loop from, not restart at 0.
- `best_val_loss` — a plain float, needed by Part 4's "only save when improved" logic.

`torch.save` can serialize this whole dict (tensors and plain Python values mixed together) to one file in one call.

In [2]:
def save_checkpoint(path, model, optimizer, epoch, best_val_loss):
    # TODO: build the checkpoint dict and torch.save it to `path`
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "best_val_loss": best_val_loss
    }
    torch.save(checkpoint, path)
    

save_checkpoint(ckpt_dir / "test.pt", model, optimizer, epoch=0, best_val_loss=float("inf"))
assert (ckpt_dir / "test.pt").exists()
print("save_checkpoint OK")

save_checkpoint OK


## Part 3: Loading and resuming

TODO: write `load_checkpoint(path, model, optimizer)` returning `(epoch, best_val_loss)`.

The pattern: `torch.load` the dict back, then hand each state dict to the *matching already-constructed* object's own `.load_state_dict(...)` method — `model` and `optimizer` need to already exist (same architecture/hyperparameters as when saved) before you can load into them; you're restoring their internal state, not reconstructing them from scratch.

In [3]:
def load_checkpoint(path, model, optimizer):
    # TODO: torch.load the dict, load_state_dict into model and optimizer,
    # and return the epoch and best_val_loss
    # return (epoch, best_val_loss)
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return checkpoint["epoch"], checkpoint["best_val_loss"]

# self-check: save real (non-random-init) state, load into a FRESH model, verify weights match
x_probe = torch.randn(8, n_features)
with torch.no_grad():
    out_before = model(x_probe).clone()

save_checkpoint(ckpt_dir / "probe.pt", model, optimizer, epoch=3, best_val_loss=0.55)

fresh_model = nn.Sequential(nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, 1))
fresh_optimizer = torch.optim.Adam(fresh_model.parameters(), lr=1e-2)
loaded_epoch, loaded_best = load_checkpoint(ckpt_dir / "probe.pt", fresh_model, fresh_optimizer)

with torch.no_grad():
    out_after = fresh_model(x_probe)

assert torch.allclose(out_before, out_after), "loaded model doesn't match saved model's outputs"
assert loaded_epoch == 3 and loaded_best == 0.55
print("load_checkpoint OK -- outputs match, epoch/best_val_loss restored")

load_checkpoint OK -- outputs match, epoch/best_val_loss restored


## Part 4: Save-best-only training loop

TODO: a training loop that only calls `save_checkpoint` when validation loss **improves** on the best seen so far — the standard "best model" checkpointing pattern, as opposed to saving every single epoch regardless of whether it helped.

You'll need a train/val split — reuse the `stratified_split` idea from Week 2 Day 1, or a plain `random_split`, either is fine here since this dataset isn't imbalanced.

In [7]:
from torch.utils.data import TensorDataset, DataLoader, random_split

ds = TensorDataset(X, y)
gen = torch.Generator().manual_seed(0)
train_ds, val_ds = random_split(ds, [0.8, 0.2], generator=gen)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

best_val_loss = float("inf")
n_epochs = 30

for epoch in range(n_epochs):
    model.train()
    for xb, yb in train_loader:
        # TODO: standard training step (zero_grad, forward, loss, backward, step)
        optimizer.zero_grad()
        output = model(xb).squeeze(1)  # Ensure output shape matches yb
        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()
       

    model.eval()
    val_loss_total, n_val = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            # TODO: accumulate val loss the same way Day 5's training loop did (weighted by batch size)
            output = model(xb).squeeze(1)  # Ensure output shape matches yb
            loss = criterion(output, yb)
            batch_size = yb.size(0)
            val_loss_total += loss.item() * batch_size
            n_val += batch_size
    val_loss = val_loss_total / n_val

    # TODO: if val_loss improved on best_val_loss, update best_val_loss and save_checkpoint(...)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(ckpt_dir / "best.pt", model, optimizer, epoch, best_val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1:3d} | val_loss {val_loss:.4f} | best {best_val_loss:.4f}")

epoch  10 | val_loss 0.0542 | best 0.0542
epoch  20 | val_loss 0.0288 | best 0.0288
epoch  30 | val_loss 0.0201 | best 0.0201


## Try yourself

1. After training, construct a brand new model + optimizer, `load_checkpoint` the best saved one, and confirm its val loss matches `best_val_loss` from training.
2. Add early stopping: if `val_loss` hasn't improved for `N` consecutive epochs, break out of the loop instead of running all `n_epochs`.
3. Save a checkpoint every epoch (not just on improvement) to a separate rotating set of files (e.g. keep only the last 3), and compare disk usage against the save-best-only approach.
4. Look up what `map_location` does in `torch.load` and when you'd need it (hint: think about what happens if a checkpoint saved on a GPU machine gets loaded on a CPU-only one).